# GLDAS-2.1 NOAH — Monthly Runoff (Qs_acc, Qsb_acc, runoff_total)

Visualize the consolidated GLDAS-2.1 NOAH 0.25° monthly runoff dataset
(NASA GES DISC, `GLDAS_NOAH025_M v2.1`).

- **Primary variable:** `runoff_total` — total runoff (Qs_acc + Qsb_acc, kg m⁻² ≡ mm)
- **Also stored:** `Qs_acc` (storm surface runoff), `Qsb_acc` (baseflow–groundwater runoff)
- **Resolution:** 0.25° × 0.25° CONUS + contributing watersheds
- **Period:** 2000-01 to 2025-12 (312 months)
- **Reference:** Rodell et al., 2004, doi:10.1175/BAMS-85-3-381

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT   = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH   = DATASTORE / "gldas_noah_v21_monthly" / "gldas_noah_v21_monthly.nc"
PRIMARY_VAR = "runoff_total"   # kg m-2 (≡ mm per month)


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid:       {ds.lat.size} lat x {ds.lon.size} lon")
print(f"Variables:  {list(ds.data_vars)}")
print(f"Conventions: {ds.attrs.get('Conventions', 'N/A')}")


## Single-month total runoff map

In [ ]:
TARGET_TIME = "2000-07-01"

da = ds[PRIMARY_VAR].sel(time=TARGET_TIME, method="nearest")
actual_time = str(da.time.values)[:10]

fig, ax = plt.subplots(figsize=(14, 6))
da.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "kg m⁻² (mm)"})
ax.set_title(f"GLDAS-2.1 NOAH Total Runoff (runoff_total) — {actual_time}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {actual_time}:")
print(f"  min:  {float(da.min(skipna=True)):.2f} mm")
print(f"  max:  {float(da.max(skipna=True)):.2f} mm")
print(f"  mean: {float(da.mean(skipna=True)):.2f} mm")
print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## All three runoff variables — single timestep

In [ ]:
run_vars = {
    "runoff_total": "Total runoff (kg m⁻²)",
    "Qs_acc":       "Storm surface runoff (kg m⁻²)",
    "Qsb_acc":      "Baseflow–groundwater runoff (kg m⁻²)",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (var, label) in zip(axes, run_vars.items()):
    if var not in ds.data_vars:
        ax.set_title(f"{label}\n(not in dataset)")
        ax.set_visible(False)
        continue
    da = ds[var].sel(time=TARGET_TIME, method="nearest")
    da.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "kg m⁻²"})
    ax.set_title(f"{var}\n{label}")
    ax.set_aspect("equal")

fig.suptitle(f"GLDAS-2.1 NOAH Runoff Variables — {actual_time}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## CONUS subset — calibration period mean (2000–2009)

In [ ]:
# GLDAS lat is ascending (24.88→52.88); slice(lower, upper)
conus = ds[PRIMARY_VAR].sel(
    lat=slice(23.9, 50.1),
    lon=slice(-125.1, -65.9),
    time=slice("2000-01-01", "2009-12-31"),
)

mean_runoff = conus.mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 8))
mean_runoff.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "kg m⁻² (mm)"})
ax.set_title("Mean Total Runoff (runoff_total) — CONUS 2000–2009")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print("CONUS 2000-2009 mean stats:")
print(f"  min:  {float(mean_runoff.min(skipna=True)):.2f} mm")
print(f"  max:  {float(mean_runoff.max(skipna=True)):.2f} mm")
print(f"  mean: {float(mean_runoff.mean(skipna=True)):.2f} mm")


## Seasonal comparison (CONUS, 2000–2009)

In [ ]:
seasons = {
    "DJF (Winter)": conus.sel(time=conus.time.dt.month.isin([12, 1, 2])),
    "MAM (Spring)": conus.sel(time=conus.time.dt.month.isin([3, 4, 5])),
    "JJA (Summer)": conus.sel(time=conus.time.dt.month.isin([6, 7, 8])),
    "SON (Fall)":   conus.sel(time=conus.time.dt.month.isin([9, 10, 11])),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, seasonal_data) in zip(axes.flatten(), seasons.items()):
    seasonal_mean = seasonal_data.mean(dim="time")
    seasonal_mean.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "kg m⁻²"})
    ax.set_title(f"{label} mean")
    ax.set_aspect("equal")

fig.suptitle("Seasonal Mean Total Runoff — CONUS 2000–2009", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Monthly time series (CONUS spatial mean)

In [ ]:
conus_full = ds[PRIMARY_VAR].sel(
    lat=slice(23.9, 50.1),
    lon=slice(-125.1, -65.9),
)
monthly_mean = conus_full.mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(16, 4))
monthly_mean.plot(ax=ax, color="#2980b9", linewidth=0.5)
ax.set_ylabel("Mean Total Runoff (kg m⁻²)")
ax.set_title("CONUS-mean monthly total runoff (runoff_total) — GLDAS-2.1 NOAH")
ax.axvspan("2000-01-01", "2009-12-31", alpha=0.15, color="orange", label="Calibration period")
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.show()


## Monthly climatology (CONUS, 2000–2009)

In [ ]:
monthly_clim = conus.groupby("time.month").mean(dim=["time", "lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_clim.month, monthly_clim.values, color="#2980b9", edgecolor="white")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Total Runoff (kg m⁻²)")
ax.set_title("Monthly Climatology — CONUS 2000–2009 (runoff_total)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()


## CF compliance verification

In [ ]:
print("CF Compliance Checks:")
print(f"  Conventions attr:  {ds.attrs.get('Conventions', 'MISSING')}")
print(f"  Time dtype:        {ds.time.dtype}")
print(f"  Time first:        {ds.time.values[0]}")
print(f"  Time last:         {ds.time.values[-1]}")
print(f"  CRS variable:      {'crs' in ds.data_vars}")
if "crs" in ds.data_vars:
    print(f"  Grid mapping name: {ds['crs'].attrs.get('grid_mapping_name', 'MISSING')}")
    crs_wkt = ds['crs'].attrs.get('crs_wkt', 'MISSING')
    print(f"  CRS WKT:           {str(crs_wkt)[:60]}...")
print(f"  time_bnds:         {'time_bnds' in ds.data_vars}")
for var in ["runoff_total", "Qs_acc", "Qsb_acc"]:
    if var in ds.data_vars:
        print(f"  {var}:")
        print(f"    grid_mapping: {ds[var].attrs.get('grid_mapping', 'MISSING')}")
        print(f"    units:        {ds[var].attrs.get('units', 'MISSING')}")
        print(f"    long_name:    {ds[var].attrs.get('long_name', 'MISSING')}")
        print(f"    cell_methods: {ds[var].attrs.get('cell_methods', 'MISSING')}")


## Manifest provenance check

In [ ]:
import json as _json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = _json.loads(manifest_path.read_text())
    entry = manifest.get("sources", {}).get("gldas_noah_v21_monthly", {})
    print(f"Source key:     {entry.get('source_key', 'N/A')}")
    print(f"Period:         {entry.get('period', 'N/A')}")
    print(f"Variables:      {entry.get('variables', 'N/A')}")
    print(f"Consolidated:   {entry.get('consolidated_nc', 'N/A')}")
    print(f"Files:          {len(entry.get('files', []))}")
    print(f"Timestamp:      {entry.get('download_timestamp', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
